# Uber Drive Behavior and Productivity Analysis

**Recruiter-facing end-to-end analysis · Decision-oriented exploratory analysis · Python 3.12/3.13**

> After mixed-format date repair, 1,154 valid trips total 12,194.8 miles; 94.1% of recorded mileage is categorized as business.

## Executive summary

**Objective:** Audit and summarize a personal trip log to expose temporal, purpose, route, and distance patterns without inventing a prediction target.

**Data:** 1,155 source rows containing trip timestamps, categories, locations, distance, and purpose.

**Verified result:** After mixed-format date repair, 1,154 valid trips total 12,194.8 miles; 94.1% of recorded mileage is categorized as business.

**Decision supported:** Improve purpose capture, route scheduling, mileage review, and reimbursement evidence.

The figures, tables, metrics, and execution counts in this notebook are saved outputs from the bundled data.

## 1. Business understanding

**Primary user:** A traveler, fleet analyst, or expense-process owner.

**Decision:** Improve purpose capture, route scheduling, mileage review, and reimbursement evidence.

**Why it matters:** A technically accurate result is useful only when its error costs, uncertainty, and decision boundary are visible. This project stays within the evidence available in the source data.

## 2. Analytical objective and success criteria

Technical success requires portable execution, explicit data-quality evidence, a justified baseline, leakage-safe validation, task-appropriate metrics, diagnostics, and saved artifacts. Business success requires a specific recommendation supported by the observed result without invented financial impact.

## 3. Reproducible environment

In [1]:
from pathlib import Path
import hashlib, importlib.util, json, os, platform, tempfile, time
os.environ.setdefault("MPLCONFIGDIR", str(Path(tempfile.gettempdir()) / "portfolio-matplotlib-cache"))
import matplotlib, numpy as np, pandas as pd, scipy, sklearn
SLUG = '02-uber-drive-analysis'
def locate_project():
    for base in [Path.cwd().resolve(), *Path.cwd().resolve().parents]:
        candidate = base if base.name == SLUG else base / "projects" / SLUG
        if (candidate / "src" / "analysis.py").is_file(): return candidate
    raise FileNotFoundError(SLUG)
PROJECT_ROOT = locate_project(); DATA_DIR = PROJECT_ROOT / "data"; REPORTS_DIR = PROJECT_ROOT / "reports"
print(pd.Series({"Python": platform.python_version(), "pandas": pd.__version__, "NumPy": np.__version__, "SciPy": scipy.__version__, "scikit-learn": sklearn.__version__, "Matplotlib": matplotlib.__version__}, name="version").to_string())
print(f"\nProject: {PROJECT_ROOT.name}")

Python          3.12.10
pandas            2.3.3
NumPy             2.5.2
SciPy            1.18.0
scikit-learn      1.9.0
Matplotlib       3.11.1

Project: 02-uber-drive-analysis


## 4. Data provenance and scope

The dataset is bundled in the original repository; upstream collection context and reuse terms are not documented.

The next cells expose exact files, byte sizes, checksums, schemas, and sample records.

### 4.1 Source-file inventory

In [2]:
rows=[]
for path in sorted(DATA_DIR.iterdir()):
    if path.is_file() and path.name != "README.md": rows.append({"file": path.name, "size_mb": round(path.stat().st_size/1_000_000,3), "sha256": hashlib.sha256(path.read_bytes()).hexdigest()[:16]})
inventory=pd.DataFrame(rows); print(inventory.to_string(index=False))

           file  size_mb           sha256
uber_drives.csv    0.088 c3f754ae34dc4529


### 4.2 Raw-record and schema preview

In [3]:
def preview(path):
    if path.suffix.lower()==".csv": return pd.read_csv(path, nrows=5)
    if path.suffix.lower()==".xlsx": return pd.read_excel(path, nrows=5)
    return None
for path in sorted(DATA_DIR.iterdir()):
    frame=preview(path)
    if frame is None: continue
    for column in frame.select_dtypes(include="object"): frame[column]=frame[column].astype(str).str.replace(r"\\s+"," ",regex=True).str.slice(0,100)
    print(f"\n{path.name}: {frame.shape[1]} columns"); print(frame.to_string(index=False,max_cols=12))


uber_drives.csv: 7 columns
     START_DATE*        END_DATE* CATEGORY*      START*           STOP*  MILES*        PURPOSE*
01-01-2016 21:11 01-01-2016 21:17  Business Fort Pierce     Fort Pierce     5.1  Meal/Entertain
01-02-2016 01:25 01-02-2016 01:37  Business Fort Pierce     Fort Pierce     5.0             nan
01-02-2016 20:25 01-02-2016 20:38  Business Fort Pierce     Fort Pierce     4.8 Errand/Supplies
01-05-2016 17:31 01-05-2016 17:45  Business Fort Pierce     Fort Pierce     4.7         Meeting
01-06-2016 14:42 01-06-2016 15:49  Business Fort Pierce West Palm Beach    63.7  Customer Visit


## 5. Data-quality assessment

The pipeline checks missingness, duplicates, invalid fields, identifiers, cardinality, and problem-specific leakage or chronology risks. No row is silently removed.

## 6. Reusable implementation

Large functions are kept in source code so the notebook remains a readable analytical narrative.

In [4]:
source_path=PROJECT_ROOT/"src"/"analysis.py"; text=source_path.read_text(encoding="utf-8")
print(f"Reusable implementation: {len(text.splitlines())} lines")
print("Functions:", ", ".join(line.split("(")[0].replace("def ","").strip() for line in text.splitlines() if line.startswith("def ")))

Reusable implementation: 77 lines
Functions: run_analysis


## 7. Methodology and hypotheses

Mixed-format datetime repair, data-quality rules, duration and speed checks, temporal aggregation, route concentration, missing-purpose analysis, and IQR outlier review.

The central hypothesis is that the audited features or group structure contain decision-relevant signal beyond the documented baseline. Exploratory findings are not presented as causal effects.

## 8. Execute the complete pipeline

This cell reruns cleaning, feature engineering, model/statistical analysis, validation, tables, figures, and model artifacts.

In [5]:
spec=importlib.util.spec_from_file_location("rebuilt_02_uber_drive_analysis", PROJECT_ROOT/"src"/"analysis.py")
analysis=importlib.util.module_from_spec(spec); spec.loader.exec_module(analysis)
started=time.perf_counter(); results=analysis.run_analysis(); runtime=time.perf_counter()-started
assert results["status"]=="passed"
print(f"Pipeline status: {results['status']}\nRuntime: {runtime:.2f} seconds")

Pipeline status: passed
Runtime: 0.41 seconds


## 9. Executed data-quality evidence

In [6]:
for path in sorted((REPORTS_DIR/"tables").glob("*data_quality.csv")):
    frame=pd.read_csv(path); print(f"\n{path.name} ({len(frame)} fields)"); print(frame.to_string(index=False,max_rows=30))


data_quality.csv (7 fields)
     column   dtype  missing_count  missing_percent  unique_values  constant
START_DATE*  object              0            0.000           1154     False
  END_DATE*  object              0            0.000           1154     False
  CATEGORY*  object              0            0.000              2     False
     START*  object              0            0.000            176     False
      STOP*  object              0            0.000            187     False
     MILES* float64              0            0.000            256     False
   PURPOSE*  object            502           43.463             11     False


## 10. Baseline, candidates, and primary result

In [7]:
primary=REPORTS_DIR/"tables"/'monthly_trip_summary.csv'
frame=pd.read_csv(primary); print(f"Primary evidence: {primary.name}, shape={frame.shape}")
print(frame.head(15).round(4).to_string(index=False))
print("\nVerified result:\n" + 'After mixed-format date repair, 1,154 valid trips total 12,194.8 miles; 94.1% of recorded mileage is categorized as business.')

Primary evidence: monthly_trip_summary.csv, shape=(12, 4)
  month  trips  miles  median_miles
2016-01     61  512.9          5.50
2016-02    115  908.2          6.10
2016-03    113 1693.9          6.60
2016-04     54 1113.0          8.80
2016-05     49  363.8          6.10
2016-06    107  832.9          7.20
2016-07    112 1224.6          7.10
2016-08    133 1335.5          5.70
2016-09     36  601.8          9.70
2016-10    106 1810.0          8.35
2016-11    122  816.9          3.45
2016-12    146  981.3          4.50

Verified result:
After mixed-format date repair, 1,154 valid trips total 12,194.8 miles; 94.1% of recorded mileage is categorized as business.


## 11. Validation, diagnostics, and robustness

In [8]:
sections=[key for key in ["validation","model_selection","tuning","residual_diagnostics","outlier_sensitivity","participant_bootstrap_intervals","diagnostic_90_percent_interval","empirical_90_percent_interval"] if key in results]
for key in sections: print(f"\n{key.upper()}\n"+json.dumps(results[key],indent=2)[:6000])

## 12. Visual evidence

### Miles By Category

![miles_by_category](../reports/figures/miles_by_category.png)

### Uber Trip Behavior Evidence

![uber_trip_behavior_evidence](../reports/figures/uber_trip_behavior_evidence.png)

## 13. Business interpretation

After mixed-format date repair, 1,154 valid trips total 12,194.8 miles; 94.1% of recorded mileage is categorized as business.

The correct action is to use this result as evidence for **Improve purpose capture, route scheduling, mileage review, and reimbursement evidence.**, while retaining the documented baseline and monitoring the error or sensitivity segments.

## 14. Prioritized recommendations

1. Use the verified result to define a controlled follow-up rather than an automatic decision.
2. Monitor the weakest subgroup, time window, interval coverage, or cluster sensitivity shown in the saved tables.
3. Revalidate against a transparent baseline whenever the data or operating context changes.

## 15. Limitations, ethics, and responsible use

This is one personal log over a limited period; location and purpose data are sensitive and findings do not generalize to platform-wide demand.

Automated outputs remain associative unless a causal study design says otherwise.

## 16. Saved-artifact integrity

In [9]:
rows=[]
for path in sorted(REPORTS_DIR.rglob("*")):
    if path.is_file(): rows.append({"artifact": str(path.relative_to(PROJECT_ROOT)), "size_kb": round(path.stat().st_size/1000,1), "sha256": hashlib.sha256(path.read_bytes()).hexdigest()[:12]})
artifacts=pd.DataFrame(rows); print(artifacts.to_string(index=False,max_rows=80))

                                       artifact  size_kb       sha256
          reports\figures\miles_by_category.png     32.8 0a212fbda1f1
reports\figures\uber_trip_behavior_evidence.png    212.4 d63bcf76285d
                           reports\metrics.json      1.4 0f2c6d5c0518
                reports\tables\data_quality.csv      0.3 1efb4dd92402
           reports\tables\distance_outliers.csv     12.7 c43f98400c38
        reports\tables\monthly_trip_summary.csv      0.3 41c69310c6e4
             reports\tables\purpose_summary.csv      0.3 88c13663960c
                  reports\tables\top_routes.csv      0.5 6f07e44ad1ed
             reports\tables\weekday_summary.csv      0.2 b3576cbd5984


## 17. Acceptance check

In [10]:
metrics=json.loads((REPORTS_DIR/"metrics.json").read_text(encoding="utf-8"))
assert metrics["status"]=="passed"
assert list((REPORTS_DIR/"figures").glob("*.png"))
assert list((REPORTS_DIR/"tables").glob("*.csv"))
assert all(path.stat().st_size>0 for path in REPORTS_DIR.rglob("*") if path.is_file())
print("PASS: metrics status, figures, tables, and non-empty artifacts verified")

PASS: metrics status, figures, tables, and non-empty artifacts verified


## 18. Conclusion

The project addressed audit and summarize a personal trip log to expose temporal, purpose, route, and distance patterns without inventing a prediction target. using mixed-format datetime repair, data-quality rules, duration and speed checks, temporal aggregation, route concentration, missing-purpose analysis, and iqr outlier review. The final verified conclusion is: **After mixed-format date repair, 1,154 valid trips total 12,194.8 miles; 94.1% of recorded mileage is categorized as business.** The next responsible step is external or current-data validation before operational use.

## 19. Reproduce locally

```bash
python projects/02-uber-drive-analysis/src/analysis.py
python scripts/execute_notebooks.py --project 02-uber-drive-analysis
```